In [ ]:
import os, sys

PROJECT_ROOT = os.getcwd()  # /home/.../PY_Finance_RL_Project
FINRL_ROOT = os.path.join(PROJECT_ROOT, "src", "FinRL")
sys.path.append(FINRL_ROOT)

print("FINRL_ROOT:", FINRL_ROOT)

In [ ]:
import os
import pandas as pd
print(os.getcwd())
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader 
import matplotlib.pyplot as plt
%matplotlib inline
from envs.gym_portfolio_env import SimplePortfolioEnv , GymPortfolioEnv 

In [ ]:

## Import the stock data
TICKERS = ['AAPL', 'MSFT', 'TSLA', 'AMZN', 'BLK']

START_DATE = '2005-01-01'
END_DATE   = '2025-01-01'

## Loading the historical datva 
df: pd.DataFrame = YahooDownloader(
                      start_date = START_DATE,
                      end_date = END_DATE,
                      ticker_list = TICKERS

                     ).fetch_data()

print(f"Raw data shape : {df.shape}")
print(df.head())



In [ ]:
## The data preprocessing

price_df : pd.DataFrame = df.pivot(index='date', columns='tic', values='close')
# This way is transsformer the pivot table to the normal table, and the index is the date, the columns is the tic, and the values is the close price.

price_df = price_df.sort_index()

print(price_df)
## Normalized price data
normalized_price_df = price_df / price_df.iloc[0]

# Draw the normalized price data
plt.figure(figsize=(12, 6))

for tic in TICKERS:
    plt.plot(normalized_price_df.index, normalized_price_df[tic], label=tic)

plt.legend()
plt.title('Normalized Stock Prices (start = 1)')

# The data x fillter (only used the 250 days data)
xticks_idx = range(0, len(normalized_price_df.index), 250)
xticks_labels = [normalized_price_df.index[i] for i in xticks_idx]
plt.xticks(xticks_idx, xticks_labels, rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# date index 
train_start = "2005-01-01"
train_end = "2005-12-31"
valid_start = "2016-01-01"
valid_end = "2020-12-31"

# the data split
price_df_train = price_df.loc[train_start:train_end].copy()
price_df_valid= price_df.loc[valid_start:valid_end].copy()

print(f"price_df_train : {price_df_train} , price_df_valid : {price_df_valid}  ")

In [ ]:
from envs.gym_portfolio_env import PortfolioEnvConfig, GymPortfolioEnv

config = PortfolioEnvConfig(tickers=TICKERS, init_wealth=1.0)
env_train = GymPortfolioEnv(price_df_train, config)

obs, info = env_train.reset()
print("obs shape after reset:", obs.shape)

for _ in range(5):
    action = env_train.action_space.sample()
    obs, reward, terminated, truncated, info = env_train.step(action)
    print("obs shape in step:", obs.shape)
    if terminated or truncated:
        break

print("MACD window:", list(env_train.macd_window))
print("BB dev window:", list(env_train.bb_dev_window))

In [ ]:
from envs.gym_portfolio_env import PortfolioEnvConfig, GymPortfolioEnv

config = PortfolioEnvConfig(TICKERS, 1.0)

env_train = GymPortfolioEnv(price_df_train, config)
env_valid = GymPortfolioEnv(price_df_valid, config)

In [ ]:
import torch
import numpy as np
from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n_actions = env_train.action_space.shape[-1]

# OU noise / Gaussian noise：鼓勵探索
action_noise = NormalActionNoise(
    mean=np.zeros(n_actions),
    sigma=0.1 * np.ones(n_actions)
)

model = DDPG(
    "MlpPolicy",
    env_train,
    action_noise=action_noise,
    verbose=1,
    learning_rate=3e-4,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    train_freq=(1, "step"),
    gradient_steps=1,
    device=device,
)

In [ ]:
model.learn(total_timesteps=1000000)

In [ ]:
model.save("ddpg_portfolio_offline")

In [ ]:
# 重設 valid env
obs, info = env_valid.reset()
done = False

wealth_traj = [env_valid.core_env.current_wealth]
returns_traj = []
rewards_traj = []

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env_valid.step(action)
    
    wealth_traj.append(info["wealth"])
    returns_traj.append(info["portfolio_return"])
    rewards_traj.append(info["rolling_reward"])

    done = terminated or truncated

In [ ]:
wealth_series = pd.Series(
    wealth_traj,
    index=price_df_valid.index[:len(wealth_traj)]
)

import matplotlib.pyplot as plt
plt.figure(figsize=(12, 6))
plt.plot(wealth_series.index, wealth_series.values, label="DDPG Strategy")

plt.title("Out-of-sample Wealth Curve (Valid)")
plt.xlabel("Date")
plt.ylabel("Wealth")
plt.legend()
plt.show()

In [ ]:
model.learn(total_timesteps=1000000)

# online

In [46]:
from datetime import datetime, timedelta

# 今天當作 end_date
online_start_str = "2026-01-01"
online_end_str   =  datetime.today().date()

# 往前推 180 天當作 start_date（你可以自己改天數）
# 例如: 2024-12-10 2025-06-07

In [47]:
data_online = YahooDownloader(
    start_date = online_start_str,
    end_date   = online_end_str,
    ticker_list = TICKERS
).fetch_data()

data_online["date"] = pd.to_datetime(data_online["date"])

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (535, 8)


In [48]:
price_df_online = (
    data_online.pivot(index="date", columns="tic", values="close")
               [TICKERS]
               .sort_index()
               .dropna(how="any")
)

print("price_df_online range:", price_df_online.index.min(), "->", price_df_online.index.max())
print("price_df_online shape:", price_df_online.shape)

price_df_online range: 2026-01-02 00:00:00 -> 2026-06-05 00:00:00
price_df_online shape: (107, 5)


In [49]:
config_online = PortfolioEnvConfig(
    tickers=TICKERS,
    init_wealth=1.0,
    wealth_norm_factor=100.0,
)

env_online = GymPortfolioEnv(price_df_online,config_online)

model = DDPG.load("ddpg_portfolio_offline", env=env_online)

obs, info = env_online.reset()
done = False
logs = []
    

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env_online.step(action)
    done = terminated or truncated

    logs.append({
        "time": env_online.core_env.price_df.index[env_online.core_env.current_step],
        "wealth": info["wealth"],
        "reward": reward,
        "drawdown": info["drawdown"],
        "action": action,
    })


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [50]:
# 從 wealth 算每日報酬
df_logs = pd.DataFrame(logs)
df_logs["time"] = pd.to_datetime(df_logs["time"])
df_logs = df_logs.set_index("time")


df_logs["daily_return"] = df_logs["wealth"].pct_change()

print("df_logs range:", df_logs.index.min(), "->", df_logs.index.max())
print(df_logs.head())

df_logs range: 2026-01-05 00:00:00 -> 2026-06-05 00:00:00
              wealth    reward  drawdown  \
time                                       
2026-01-05  1.013971  0.013971  0.000000   
2026-01-06  1.016999  0.002987  0.000000   
2026-01-07  1.016241 -0.001491  0.000745   
2026-01-08  1.019622  0.003327  0.000000   
2026-01-09  1.027230  0.007462  0.000000   

                                                       action  daily_return  
time                                                                         
2026-01-05  [0.96994525, 0.26044744, 0.9071791, 0.99618006...           NaN  
2026-01-06  [0.9075797, 0.7723998, 0.15942788, 0.6333781, ...      0.002987  
2026-01-07  [0.9736085, 0.6581658, 0.82794833, 0.98304605,...     -0.000745  
2026-01-08  [0.9986496, 0.45221114, 0.14318141, 0.84860045...      0.003327  
2026-01-09  [0.706959, 0.9081764, 0.95403653, 0.643224, 0....      0.007462  


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(
    df_logs.index,
    df_logs["reward"],
    label="Reward (Sharpe-like - DD penalty)",
)
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Date")
plt.ylabel("Reward")
plt.title("Online Simulation: Step-wise Reward (2026-01-01 ~ 2026-06-07)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# ===== 圖 2：Wealth 曲線（2026-01-01 ~ 2026-06-07） =====
plt.figure(figsize=(10, 4))
plt.plot(
    df_logs.index,
    df_logs["wealth"],
    label="Portfolio Wealth",
)
plt.xlabel("Date")
plt.ylabel("Wealth")
plt.title("Online Simulation: Portfolio Wealth (2026-01-01 ~ 2026-06-07)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# ===== 圖 3：Daily Portfolio Return（2026-01-01 ~ 2026-06-07） =====
plt.figure(figsize=(10, 4))
plt.plot(
    df_logs.index,
    df_logs["daily_return"],
    label="Daily Portfolio Return",
)
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Date")
plt.ylabel("Daily Return")
plt.title("Online Simulation: Daily Portfolio Returns (2026-01-01 ~ 2026-06-07)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()